# Task 3 — Notebook 03: Maps, time series, Bayesian diagnostics, state×crop table, run bundle

**Inputs:** `smap_anomaly_{event_id}.parquet` (with NIG Bayesian scores from NB02), CDL spatial metadata, Corn Belt state boundaries.

**Figures (per event, 6 total):**
1. Four z-score maps at spread ISO weeks (frequentist baseline)
2. Mean z time series with ±1σ band
3. Duration / persistence map (wet or dry fraction)
4. **NIG posterior predictive drought exceedance** — 4-panel `nig_p_drought` maps (percentile-scale, RdYlBu)
5. **NIG posterior uncertainty** — single map of `nig_posterior_scale` showing epistemically honest regions
6. **z-score vs NIG scatter** — frequentist z vs −log₁₀(nig_p_anomaly), colored by posterior scale

**Tables:** `artifacts/tables/task3/task3__{event_id}__anomaly_stats_by_state_crop__*.csv` now include `mean_nig_p_drought` and `frac_pdrought_lt_0p1`.

**Run log:** `artifacts/logs/runs/<id>/run_bundle.json` lists all paths by event. `event_id`.


In [1]:
import json
import sys
import uuid
from datetime import date, datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Repo root not found")
sys.path.insert(0, str(REPO_ROOT))

from src.io.cdl_parquet import load_cdl_spatial_metadata
from src.modeling.task3_aggregate import attach_state_name, state_crop_anomaly_summary
from src.modeling.task3_smap_anomalies import event_windows_from_cfg
from src.viz.rotation_maps import load_cornbelt_state_boundaries_5070
from src.viz.task3_maps import fill_raster, plot_extent_from_metadata, plot_z_map

with open(REPO_ROOT / "configs" / "task3_soil_moisture.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

events = event_windows_from_cfg(cfg)
meta = load_cdl_spatial_metadata(REPO_ROOT)
h, w = int(meta["height"]), int(meta["width"])
extent = plot_extent_from_metadata(meta)
states = load_cornbelt_state_boundaries_5070(REPO_ROOT)

fig_dir = REPO_ROOT / cfg["output"]["figures_dir"]
tbl_dir = REPO_ROOT / cfg["output"]["tables_dir"]
fig_dir.mkdir(parents=True, exist_ok=True)
tbl_dir.mkdir(parents=True, exist_ok=True)
date_s = date.today().strftime("%Y%m%d")

out_root = REPO_ROOT / cfg["output"]["processed_dir"]
outputs_by_event: dict[str, dict] = {}


def panel_iso_weeks(iso_series: pd.Series) -> list[int]:
    wks = sorted({int(w) for w in iso_series.dropna().unique()})
    n = len(wks)
    if n == 0:
        return []
    if n <= 4:
        return wks
    idx = np.linspace(0, n - 1, num=4, dtype=float)
    pick = sorted({int(round(i)) for i in idx})
    return [wks[i] for i in pick]


def duration_fraction_by_pixel(anom: pd.DataFrame, mode: str, z_thr: float) -> pd.DataFrame:
    z = anom["z_score"].to_numpy()
    if mode == "dry_below":
        flag = z < float(z_thr)
    else:
        flag = z > float(z_thr)
    x = anom.assign(_hit=flag)
    return x.groupby(["iy", "ix"], as_index=False)["_hit"].mean().rename(columns={"_hit": "frac_weeks"})


for ev in events:
    eid = str(ev["id"])
    label = str(ev.get("label", eid))
    mode = str(ev.get("duration_mode", "wet_above"))
    z_thr = float(ev.get("duration_z", 1.5 if mode != "dry_below" else -1.5))

    anom_pq = out_root / f"smap_anomaly_{eid}.parquet"
    if not anom_pq.is_file():
        raise FileNotFoundError(f"Run notebook 02 first: missing {anom_pq}")
    anom = pd.read_parquet(anom_pq)
    anom = attach_state_name(REPO_ROOT, anom)

    has_nig = "nig_p_drought" in anom.columns

    # ---- FIGURE 1: z-score 4-panel maps ----
    targets = panel_iso_weeks(anom["iso_week"])
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), dpi=120)
    axes = axes.ravel()
    for ax, tw in zip(axes, targets):
        sub = anom.loc[anom["iso_week"] == tw].drop_duplicates(subset=["iy", "ix"])
        if sub.empty:
            ax.set_axis_off()
            continue
        g = fill_raster(h, w, sub["iy"].to_numpy(), sub["ix"].to_numpy(), sub["z_score"].to_numpy())
        ax.imshow(g, extent=list(extent), origin="upper", interpolation="nearest", cmap="RdBu_r", vmin=-3, vmax=3)
        if states is not None and not states.empty:
            states.boundary.plot(ax=ax, color="#222", linewidth=0.6)
        d0 = sub["date"].iloc[0]
        ax.set_title(f"ISO week {tw} ({d0})")
        ax.set_axis_off()
    fig.suptitle(f"{label} — SMAP soil moisture z-score (vs 2015–2021 weekly climatology)", fontsize=11)
    fig.tight_layout()
    p4 = fig_dir / f"task3__{eid}__anomaly_map_4panel__{date_s}.png"
    fig.savefig(p4, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved", p4.relative_to(REPO_ROOT))

    # ---- FIGURE 2: time series ----
    ts = anom.groupby("date", as_index=False).agg(
        mean_z=("z_score", "mean"), std_z=("z_score", "std"), n=("z_score", "count")
    )
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    x = np.arange(len(ts))
    ax2.plot(x, ts["mean_z"], color="#2980b9", lw=2)
    ax2.fill_between(x, ts["mean_z"] - ts["std_z"], ts["mean_z"] + ts["std_z"], alpha=0.25, color="#2980b9")
    ax2.axhline(0, color="k", lw=0.6)
    ax2.axhline(1, color="g", ls="--", lw=0.7)
    ax2.axhline(-1, color="orange", ls="--", lw=0.7)
    ax2.set_xticks(x)
    ax2.set_xticklabels(ts["date"], rotation=45, ha="right", fontsize=7)
    ax2.set_ylabel("mean z (rotation-eligible pixels)")
    ax2.set_title(f"{eid} — area-mean soil moisture anomaly")
    fig2.tight_layout()
    pts = fig_dir / f"task3__{eid}__anomaly_timeseries_cropland__{date_s}.png"
    fig2.savefig(pts, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved", pts.relative_to(REPO_ROOT))

    # ---- FIGURE 3: duration / persistence ----
    cnt = duration_fraction_by_pixel(anom, mode, z_thr)
    g3 = fill_raster(h, w, cnt["iy"].to_numpy(), cnt["ix"].to_numpy(), cnt["frac_weeks"].to_numpy())
    if mode == "dry_below":
        dur_title = f"Fraction of event weeks with z < {z_thr}"
        dur_cmap = "YlOrBr"
    else:
        dur_title = f"Fraction of event weeks with z > {z_thr}"
        dur_cmap = "Blues"
    fig3, ax3 = plot_z_map(g3, extent, title=dur_title, vmin=0, vmax=1, cmap=dur_cmap)
    dur_png = fig_dir / f"task3__{eid}__duration_fraction__{date_s}.png"
    fig3.savefig(dur_png, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved", dur_png.relative_to(REPO_ROOT))

    # ---- FIGURE 4 (NIG): posterior predictive drought exceedance 4-panel ----
    nig_figs: dict[str, str] = {}
    if has_nig:
        fig4, axes4 = plt.subplots(2, 2, figsize=(12, 10), dpi=120)
        axes4 = axes4.ravel()
        for ax, tw in zip(axes4, targets):
            sub = anom.loc[anom["iso_week"] == tw].drop_duplicates(subset=["iy", "ix"])
            if sub.empty:
                ax.set_axis_off()
                continue
            g4 = fill_raster(h, w, sub["iy"].to_numpy(), sub["ix"].to_numpy(), sub["nig_p_drought"].to_numpy())
            im4 = ax.imshow(
                g4, extent=list(extent), origin="upper", interpolation="nearest",
                cmap="RdYlBu", vmin=0.0, vmax=1.0,
            )
            if states is not None and not states.empty:
                states.boundary.plot(ax=ax, color="#222", linewidth=0.6)
            d0 = sub["date"].iloc[0]
            ax.set_title(f"ISO week {tw} ({d0})")
            ax.set_axis_off()
        fig4.suptitle(
            f"{label} — NIG posterior predictive P(drought exceedance)\n"
            "0 = extremely dry  |  0.5 = normal  |  1 = extremely wet",
            fontsize=10,
        )
        fig4.colorbar(im4, ax=axes4.tolist(), fraction=0.025, pad=0.02, label="P(drought)")
        fig4.tight_layout(rect=[0, 0, 0.92, 0.95])
        nig4_png = fig_dir / f"task3__{eid}__nig_p_drought_4panel__{date_s}.png"
        fig4.savefig(nig4_png, dpi=200, bbox_inches="tight")
        plt.show()
        print("Saved", nig4_png.relative_to(REPO_ROOT))
        nig_figs["figures_nig_p_drought"] = str(nig4_png.relative_to(REPO_ROOT)).replace("\\", "/")

        # ---- FIGURE 5 (NIG): posterior uncertainty map ----
        unc = anom.groupby(["iy", "ix"], as_index=False)["nig_posterior_scale"].mean()
        g5 = fill_raster(h, w, unc["iy"].to_numpy(), unc["ix"].to_numpy(), unc["nig_posterior_scale"].to_numpy())
        fig5, ax5 = plt.subplots(figsize=(9, 7), dpi=120)
        im5 = ax5.imshow(
            g5, extent=list(extent), origin="upper", interpolation="nearest",
            cmap="magma", vmin=0.0, vmax=float(np.nanpercentile(g5[np.isfinite(g5)], 98)),
        )
        if states is not None and not states.empty:
            states.boundary.plot(ax=ax5, color="white", linewidth=0.7, alpha=0.7)
        ax5.set_title(f"{label} — NIG posterior predictive scale (epistemic uncertainty)", fontsize=10)
        ax5.set_axis_off()
        plt.colorbar(im5, ax=ax5, fraction=0.035, pad=0.02, label="predictive std (m³/m³)")
        fig5.tight_layout()
        nig5_png = fig_dir / f"task3__{eid}__nig_uncertainty__{date_s}.png"
        fig5.savefig(nig5_png, dpi=200, bbox_inches="tight")
        plt.show()
        print("Saved", nig5_png.relative_to(REPO_ROOT))
        nig_figs["figures_nig_uncertainty"] = str(nig5_png.relative_to(REPO_ROOT)).replace("\\", "/")

        # ---- FIGURE 6 (NIG): z-score vs NIG scatter ----
        peak_wk = anom.groupby("iso_week")["z_score"].apply(lambda s: s.abs().mean()).idxmax()
        peak = anom.loc[anom["iso_week"] == peak_wk].drop_duplicates(subset=["iy", "ix"])
        nlog_p = -np.log10(np.clip(peak["nig_p_anomaly"].to_numpy(), 1e-12, 1.0))
        fig6, ax6 = plt.subplots(figsize=(8, 6), dpi=120)
        sc = ax6.scatter(
            peak["z_score"].to_numpy(), nlog_p,
            c=peak["nig_posterior_scale"].to_numpy(),
            s=1, alpha=0.3, cmap="viridis", rasterized=True,
        )
        ax6.set_xlabel("Frequentist z-score")
        ax6.set_ylabel("−log₁₀(NIG two-tailed p-value)")
        ax6.set_title(f"{eid} — z-score vs Bayesian anomaly (peak week {peak_wk})", fontsize=10)
        ax6.axvline(0, color="k", lw=0.5)
        ax6.axhline(-np.log10(0.05), color="red", ls="--", lw=0.7, label="p = 0.05")
        ax6.legend(loc="upper left", fontsize=8)
        plt.colorbar(sc, ax=ax6, fraction=0.035, pad=0.02, label="NIG posterior scale")
        fig6.tight_layout()
        nig6_png = fig_dir / f"task3__{eid}__zscore_vs_nig_scatter__{date_s}.png"
        fig6.savefig(nig6_png, dpi=200, bbox_inches="tight")
        plt.show()
        print("Saved", nig6_png.relative_to(REPO_ROOT))
        nig_figs["figures_zscore_vs_nig"] = str(nig6_png.relative_to(REPO_ROOT)).replace("\\", "/")
    else:
        print(f"[{eid}] No NIG columns — re-run NB02 to add Bayesian scores.")

    # ---- STATE × CROP TABLE (now includes NIG columns when available) ----
    tab = state_crop_anomaly_summary(anom)
    csv_path = tbl_dir / f"task3__{eid}__anomaly_stats_by_state_crop__{date_s}.csv"
    tab.to_csv(csv_path, index=False)
    print("Wrote", csv_path.relative_to(REPO_ROOT))

    outputs_by_event[eid] = {
        "label": label,
        "duration_mode": mode,
        "duration_z": z_thr,
        "smap_anomaly_parquet": str(anom_pq.relative_to(REPO_ROOT)).replace("\\", "/"),
        "figures_4panel": str(p4.relative_to(REPO_ROOT)).replace("\\", "/"),
        "figures_timeseries": str(pts.relative_to(REPO_ROOT)).replace("\\", "/"),
        "figures_duration": str(dur_png.relative_to(REPO_ROOT)).replace("\\", "/"),
        "table_state_crop": str(csv_path.relative_to(REPO_ROOT)).replace("\\", "/"),
        **nig_figs,
    }

run_id = uuid.uuid4().hex[:8]
run_dir = REPO_ROOT / "artifacts" / "logs" / "runs" / run_id
run_dir.mkdir(parents=True, exist_ok=True)
bundle = {
    "task": "task3_soil_moisture",
    "run_id": run_id,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "config_path": "configs/task3_soil_moisture.yaml",
    "outputs_by_event": outputs_by_event,
}
(run_dir / "run_bundle.json").write_text(json.dumps(bundle, indent=2), encoding="utf-8")
print("Wrote", (run_dir / "run_bundle.json").relative_to(REPO_ROOT))


KeyboardInterrupt: 